# Fine-tune Hinglish Language Switch LLM (GPU)
This notebook is a reproducible GPU-run path. It does **not** contain private call data. Upload a CSV you have permission to use, with `text,label,call_id,reviewed` columns. Every row must be human-reviewed before using it for a credible evaluation.

Labels: `HINGLISH`, `ENGLISH`, `NEUTRAL`, `UNCERTAIN`. Do not claim >96% unless a representative held-out human-reviewed test set supports it.

In [ ]:
!git clone https://github.com/abhinavnautiyalDS/hinglish-language-switch-llm.git
%cd hinglish-language-switch-llm
!pip install -U transformers peft accelerate scikit-learn pandas

## Upload the reviewed dataset
Use the private review pack locally, correct labels in a CSV, and upload only the version you are authorized to process. Do not upload the raw transcript to the public GitHub repo.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select your human-reviewed CSV, e.g. utterances_reviewed.csv
DATA_PATH = next(iter(uploaded.keys()))
print('Using:', DATA_PATH)

In [ ]:
import pandas as pd
df = pd.read_csv(DATA_PATH)
display(df.head())
assert {'text','label','call_id','reviewed'}.issubset(df.columns)
assert df['reviewed'].astype(str).str.lower().isin({'true','1','yes','y'}).all(), 'Every row must be human-reviewed for this benchmark.'
assert set(df['label'].astype(str).str.upper()).issubset({'HINGLISH','ENGLISH','NEUTRAL','UNCERTAIN'})
print('Rows:', len(df), 'labels:', df['label'].value_counts().to_dict(), 'calls:', df['call_id'].nunique())

In [ ]:
!python src/prepare_data.py --input "$DATA_PATH" --output-dir data/splits --require-reviewed

## Fine-tune Qwen2.5-0.5B-Instruct with LoRA
Select **Runtime → Change runtime type → GPU** before training. Training is not performed automatically on GitHub.

In [ ]:
!python src/train_lora.py --train data/splits/train.csv --validation data/splits/validation.csv --output-dir artifacts/qwen-hinglish-lora --epochs 3 --batch-size 4 --grad-accumulation 4

In [ ]:
!python src/evaluate.py --test data/splits/test.csv --model artifacts/qwen-hinglish-lora --require-reviewed --output artifacts/evaluation.json
import json
print(json.dumps(json.load(open('artifacts/evaluation.json')), indent=2))

## Save the adapter and evaluation report
Download the adapter and report for private storage. Do not commit trained weights or sensitive data unless the repository's sharing/access policy explicitly allows it.

In [ ]:
!zip -qr qwen-hinglish-training-output.zip artifacts/qwen-hinglish-lora artifacts/evaluation.json
files.download('qwen-hinglish-training-output.zip')